# Exp 7 — Concise \\boxed{} Distillation

Tests whether training on concise teacher traces ending with `\\boxed{answer}` (instead of verbose `#### answer`) improves student accuracy by aligning with the base model's pretrained output style.

**Changes from Exp 3/4:**
- Teacher prompt: concise reasoning, ends with `\\boxed{number}`
- Eval prompt: asks model to end with `\\boxed{number}`
- Answer extraction: handles both `####` and `\\boxed{}` patterns

**Runtime:** T4 GPU | **Samples:** 100 traces, 100 eval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

if not os.path.exists('/content/SC4001-Group133/.git'):
    os.system('git clone https://github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133')

os.chdir('/content/SC4001-Group133')
os.system('git fetch origin main')
os.system('git reset --hard origin/main')

import importlib
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]
importlib.invalidate_caches()

!pip install -r requirements.txt -q
!pip install -e . -q

import subprocess
print(subprocess.check_output(['git', 'log', '--oneline', '-1']).decode().strip())

In [ ]:
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable T4 GPU in Runtime > Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 1 — Generate traces with concise \\boxed{} prompt

In [ ]:
# Override output paths so Exp 7 traces don't overwrite Exp 3/4 traces
import yaml
from pathlib import Path

with open('configs/distillation.yaml') as f:
    dist_cfg = yaml.safe_load(f)

dist_cfg['dataset']['max_samples'] = 500
dist_cfg['output']['raw_traces'] = 'data/traces/exp7_raw_traces.jsonl'
dist_cfg['output']['filtered_traces'] = 'data/traces/exp7_filtered_traces.jsonl'
dist_cfg['output']['dataset_dir'] = 'data/processed/exp7_distillation'

with open('configs/distillation.yaml', 'w') as f:
    yaml.safe_dump(dist_cfg, f, sort_keys=False)

print('Teacher prompt:')
print(dist_cfg['prompt_template'])
print('Output paths:')
print('  raw:     ', dist_cfg['output']['raw_traces'])
print('  filtered:', dist_cfg['output']['filtered_traces'])
print('  dataset: ', dist_cfg['output']['dataset_dir'])

In [ ]:
# Generate raw traces
from src.distillation.generate_traces import generate_traces
generate_traces()

In [ ]:
# Filter to correct traces only
from src.distillation.filter_traces import filter_traces
filter_traces()

In [ ]:
# Build HF datasets (D1 and D2)
from src.distillation.build_dataset import build_dataset
build_dataset(use_filtered=False)  # D1 — all traces
build_dataset(use_filtered=True)   # D2 — filtered traces

## Step 2 — Train student on new traces

In [ ]:
# Point finetune config to exp7 dataset and checkpoint output
import yaml

with open('configs/finetune.yaml') as f:
    ft_cfg = yaml.safe_load(f)

ft_cfg['data']['mode'] = 'distillation'  # D1 — all traces, no filtering
ft_cfg['training']['output_dir'] = 'results/checkpoints/exp7_distillation'
ft_cfg['data']['distillation_dataset_dir'] = 'data/processed/exp7_distillation'

with open('configs/finetune.yaml', 'w') as f:
    yaml.safe_dump(ft_cfg, f, sort_keys=False)

print('Training config:')
print(f"  mode:       {ft_cfg['data']['mode']}")
print(f"  output_dir: {ft_cfg['training']['output_dir']}")
print(f"  dataset:    {ft_cfg['data']['distillation_dataset_dir']}")

In [ ]:
from src.training.finetune import train
train()

## Step 3 — Evaluate and compare against Exp 3/4

In [ ]:
# Point eval config to exp7 checkpoint
import yaml

with open('configs/eval.yaml') as f:
    eval_cfg = yaml.safe_load(f)

eval_cfg['model']['checkpoint_path'] = 'results/checkpoints/exp7_distillation'
eval_cfg['gsm8k']['max_samples'] = 500

with open('configs/eval.yaml', 'w') as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False)

print('Eval config:')
print(f"  checkpoint: {eval_cfg['model']['checkpoint_path']}")
print(f"  max_samples: {eval_cfg['gsm8k']['max_samples']}")

In [ ]:
from src.evaluation.eval_gsm8k import evaluate_gsm8k
evaluate_gsm8k(experiment_name='exp7_boxed_distillation', max_samples=500)

In [ ]:
import pandas as pd

df = pd.read_csv('results/exp7_boxed_distillation_gsm8k.csv')
acc = df['is_correct'].mean()
tokens = df['token_count'].mean()
has_boxed = df['prediction'].str.contains(r'\boxed{', regex=False).mean()
has_hash = df['prediction'].str.contains('####').mean()

print(f'Accuracy:    {acc:.1%}')
print(f'Mean tokens: {tokens:.1f}')
print(f'has \\boxed{{}}: {has_boxed:.1%}')
print(f'has ####:    {has_hash:.1%}')